<img src="https://www.funcionpublica.gov.co/documents/d/guest/logo-universidad-nacional" alt="Logo UNAL" width="600"/>

### **Universidad Nacional de Colombia sede Manizales**
#### Facultad de ingeniería y arquitectura
#### Departamento de ingeniería eléctrica, electrónica y computación
#### *Procesamiento del Lenguaje Natural*

#### Profesor: Lucas Iturriago
#### Monitora: Isabella Valero Mora - lvalerom@unal.edu.co

## Representación de Texto: De Palabras a Vectores

# Introducción: El Puente entre el Lenguaje y la Máquina
El gran desafío del Procesamiento de Lenguaje Natural (NLP) es que las computadoras no "leen" texto; solo procesan números. Por lo tanto, necesitamos un **Modelo de Espacio Vectorial** (Vector Space Model).

Representar texto consiste en transformar una secuencia de caracteres en una estructura numérica (vector) que conserve, en la medida de lo posible, la **información relevante** para una tarea (en este caso, detectar Spam).

En esta primera parte, exploraremos el enfoque más directo: la **Ingeniería de Características Manuales**.

### 1. Preparación del Entorno y Datos
Antes de modelar, necesitamos adquirir los datos y limpiarlos. Trabajaremos con el dataset *SMS Spam Collection*.

In [ ]:
import pandas as pd
import numpy as np
import urllib.request
import string
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Descarga de recursos lingüísticos esenciales de NLTK
nltk.download('stopwords')
nltk.download('punkt')

# Carga de datos desde fuente remota
url = "https://raw.githubusercontent.com/UN-GCPDS/ProcesamientoLenguajeNatural/refs/heads/main/Insumos/SMSSpamCollection.txt"
data = urllib.request.urlopen(url)

# Procesamiento de líneas: Decodificación y división por tabulaciones (TSV)
lines_split = [
    line.decode('latin-1').strip().split("\t")
    for line in data
]
df = pd.DataFrame(lines_split, columns=["label", "text"])

### 2. Preprocesamiento: Reducción de Ruido
El texto crudo tiene mucha variabilidad que no aporta valor semántico (puntuación, mayúsculas irrelevantes, palabras funcionales como "the", "a").

El objetivo aquí es la **Normalización**: llevar el texto a una forma estándar.

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """
    Realiza una limpieza básica: minúsculas, eliminación de puntuación y stopwords.
    """
    # 1. Lowercasing: Para que "Spam" y "spam" sean la misma unidad
    text = text.lower()
    
    # 2. Punctuation Removal: Elimina caracteres que suelen ser ruido
    text = ''.join([char for char in text if char not in string.punctuation])
    
    # 3. Stopwords Removal: Filtra palabras muy frecuentes pero sin peso semántico
    words = text.split()
    words = [word for word in words if word not in stop_words]
    
    return ' '.join(words)

# Aplicación del pipeline de limpieza
df['cleaned_text'] = df['text'].apply(clean_text)

# Codificación de la variable objetivo (Target encoding)
df['label_num'] = df['label'].map({'spam': 1, 'ham': 0})

print("Vista previa del dataset procesado:")
print(df[['text', 'cleaned_text', 'label_num']].head())

## 3. Ingeniería de Características Manuales (Manual Feature Engineering)

Antes de usar algoritmos complejos, podemos ser "detectives" de datos. En el Spam, la estructura suele delatar la intención. 

#
### ¿Por qué usar características manuales?
1. **Interpretabilidad:** Podemos explicar exactamente por qué un mensaje es spam (ej: "Tiene 15 números").
2. **Bajo Costo:** No requiere procesar un vocabulario de miles de palabras.
3. **Independencia del Idioma:** Muchas de estas reglas (longitud, puntuación) son universales.

Extraeremos 4 características clave:
* **message_length**: Los SMS de spam suelen intentar vender algo, por lo que suelen ser más largos.
* **caps_count**: El uso excesivo de mayúsculas indica urgencia o gritos (¡GANA YA!).
* **numbers_count**: Presencia de precios, códigos de cupón o números de contacto.
* **punctuation_count**: Uso agresivo de signos de exclamación o símbolos de moneda ($).

In [ ]:
def extract_manual_features(dataframe):
    """
    Añade columnas de metadatos estructurales al dataframe.
    """
    # Longitud total en caracteres
    dataframe['message_length'] = dataframe['text'].apply(len)
    
    # Conteo de mayúsculas (antes de la limpieza)
    dataframe['caps_count'] = dataframe['text'].apply(lambda x: sum(1 for char in x if char.isupper()))
    
    # Conteo de dígitos numéricos
    dataframe['numbers_count'] = dataframe['text'].apply(lambda x: sum(1 for char in x if char.isdigit()))
    
    # Conteo de signos de puntuación
    dataframe['punctuation_count'] = dataframe['text'].apply(lambda x: sum(1 for char in x if char in string.punctuation))
    
    return dataframe

df = extract_manual_features(df)

### 4. Evaluación del Modelo de "Línea Base" (Baseline)
Entrenaremos una Regresión Logística usando solo estos 4 números. Esto nos dirá qué tanto podemos predecir **sin siquiera leer las palabras**.

In [ ]:
features = ['message_length', 'caps_count', 'numbers_count', 'punctuation_count']
X = df[features]
y = df['label_num']

# División entrenamiento/prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenamiento del modelo
manual_model = LogisticRegression()
manual_model.fit(X_train, y_train)

# Predicción y métricas
y_pred_manual = manual_model.predict(X_test)

print("--- RENDIMIENTO CON CARACTERÍSTICAS MANUALES ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_manual):.4f}")
print("\nMatriz de confusión y métricas detalladas:")
print(classification_report(y_test, y_pred_manual))

# Representación Vectorial Clásica
En la parte anterior vimos que la estructura (longitud, mayúsculas) ayuda, pero para entender el mensaje necesitamos procesar las **palabras**. 

Aquí exploraremos la evolución desde la representación más simple (**One-Hot**) hasta la más estadística (**TF-IDF**).

---

## 1. One-Hot Encoding: El nivel atómico
La forma más básica de convertir una palabra en número es darle un "ID" único. Si nuestro vocabulario tiene 3 palabras, cada una es un vector donde solo una posición es `1` y el resto `0`.

**Limitaciones críticas:**
* **Sparsity (Dispersión):** Si el vocabulario es de 10,000 palabras, cada palabra es un vector casi vacío de 10,000 dimensiones.
* **Falta de relación:** Para este modelo, "perro" y "can" son tan diferentes como "perro" y "computadora".

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Ejemplo didáctico con una oración pequeña
sample_sentence = "go until jurong point crazy"
words = sample_sentence.split()

# Crear vocabulario ordenado
vocab = sorted(list(set(words)))
word_to_index = {word: i for i, word in enumerate(vocab)}
print(f"Vocabulario y sus índices: {word_to_index}")

# Convertir palabras a sus índices numéricos
word_indices = [word_to_index[word] for word in words]

# Configurar el codificador de Scikit-learn
encoder = OneHotEncoder(categories=[range(len(vocab))], sparse_output=False)

# Generar los vectores One-Hot
one_hot_vectors = encoder.fit_transform(np.array(word_indices).reshape(-1, 1))

print("\nRepresentación One-Hot resultante:")
for word, vector in zip(words, one_hot_vectors):
    print(f"'{word}': {vector}")

## 2. Bag of Words (BoW): La "Bolsa de Palabras"
En lugar de vectores por palabra, el **BoW** crea un vector por **documento** (o mensaje). Es simplemente un contador: ¿cuántas veces aparece cada palabra del vocabulario en este texto?



### El problema del orden
Al "meter todo en una bolsa", perdemos la sintaxis. "El perro muerde al hombre" y "El hombre muerde al perro" generan el **mismo vector BoW**, aunque el significado sea distinto.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Preparar datos usando el texto limpio de la Parte 1
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['cleaned_text'], df['label_num'], test_size=0.2, random_state=42
)

# Inicializar el vectorizador (crea la "bolsa")
bow_vectorizer = CountVectorizer()

# Aprender vocabulario y transformar entrenamiento
X_train_bow = bow_vectorizer.fit_transform(X_train_text)

# Aplicar el mismo vocabulario al set de prueba
X_test_bow = bow_vectorizer.transform(X_test_text)

print(f"Dimensiones de la matriz BoW: {X_train_bow.shape}")
print(f"Palabras en el vocabulario: {len(bow_vectorizer.get_feature_names_out())}")

# Entrenamiento y evaluación
bow_model = LogisticRegression(max_iter=1000)
bow_model.fit(X_train_bow, y_train)
y_pred_bow = bow_model.predict(X_test_bow)

print("\n--- RENDIMIENTO CON BAG OF WORDS ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_bow):.4f}")

## 3. TF-IDF: El Peso de la Relevancia
No todas las palabras valen lo mismo. En BoW, palabras muy comunes (como "el" o "y") tienen mucho peso por su frecuencia, pero no ayudan a distinguir Spam de Ham. 

**TF-IDF** soluciona esto mediante un equilibrio estadístico:



### La Matemática detrás
1. **Term Frequency (TF):** Qué tan frecuente es la palabra en el documento actual.
   $$TF(t,d) = \frac{\text{veces que } t \text{ aparece en } d}{\text{total de términos en } d}$$
2. **Inverse Document Frequency (IDF):** Qué tan "rara" es la palabra en todo el corpus. Si aparece en todos los mensajes, su importancia baja.
   $$IDF(t, D) = \log\left(\frac{|D|}{|\{d \in D : t \in d\}| + 1}\right)$$
3. **TF-IDF Result:** Es el producto de ambos.
   $$TF\text{-}IDF = TF \times IDF$$

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# El TfidfVectorizer hace todo el cálculo matemático automáticamente
tfidf_vectorizer = TfidfVectorizer()

# Transformación de los textos
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

# Entrenar el modelo sobre los pesos estadísticos
tfidf_model = LogisticRegression(max_iter=1000)
tfidf_model.fit(X_train_tfidf, y_train)
y_pred_tfidf = tfidf_model.predict(X_test_tfidf)

print("--- RENDIMIENTO CON TF-IDF ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_tfidf):.4f}")
print("\nInterpretación:")
print("TF-IDF suele superar a BoW porque penaliza palabras genéricas y resalta términos")
print("clave del Spam como 'win', 'prize' o 'claim', que son raros en mensajes normales.")

| Modelo | Representación | Ventaja | Desventaja |
| :--- | :--- | :--- | :--- |
| **One-Hot** | Binaria por palabra | Simplicidad absoluta | Memoria ineficiente, sin semántica |
| **BoW** | Conteo por documento | Fácil de entender | Ignora orden y contexto |
| **TF-IDF** | Importancia relativa | Resalta palabras clave | Sigue ignorando el orden |

# Representaciones Distribuidas
Hasta ahora, hemos tratado las palabras como unidades aisladas (BoW/TF-IDF). Sin embargo, en el lenguaje real, las palabras tienen relaciones: "feliz" está más cerca de "contento" que de "computadora".

Los **Word Embeddings** resuelven esto representando palabras como **vectores densos** de baja dimensión (normalmente entre 50 y 300).



---

## 1. Embeddings Estáticos (Word2Vec y GloVe)
Los modelos estáticos asignan un único vector fijo a cada palabra.
* **Word2Vec:** Aprende de la "compañía" de las palabras. Tiene dos arquitecturas: **CBOW** (predice palabra según contexto) y **Skip-Gram** (predice contexto según palabra).
* **GloVe (Global Vectors):** Se basa en una matriz de co-ocurrencia global en todo el corpus.



### Estrategia de Agregación
Como un mensaje tiene muchas palabras, para clasificarlo promediamos los vectores de todas sus palabras para obtener un único "vector de documento".

In [ ]:
import gensim.downloader as api

# Cargamos un modelo GloVe pre-entrenado (128MB aprox.)
# Entrenado sobre Wikipedia, mapea cada palabra a un vector de 50 dimensiones.
glove_model = api.load("glove-wiki-gigaword-50")
embedding_dim = glove_model.vector_size

def document_vector(doc):
    """
    Transforma un texto en un vector único promediando los embeddings de sus palabras.
    """
    # Filtramos palabras que no existen en el vocabulario del modelo (OOV)
    doc_words = [word for word in doc.split() if word in glove_model]
    
    if not doc_words:
        return np.zeros(embedding_dim)

    # Calculamos el promedio matemático de los vectores
    return np.mean(glove_model[doc_words], axis=0)

# Transformamos los sets de entrenamiento y prueba
X_train_glove = np.array([document_vector(doc) for doc in X_train_text])
X_test_glove = np.array([document_vector(doc) for doc in X_test_text])

# Entrenamiento del clasificador sobre vectores semánticos
glove_model_clf = LogisticRegression(max_iter=1000)
glove_model_clf.fit(X_train_glove, y_train)
y_pred_glove = glove_model_clf.predict(X_test_glove)

print("--- RENDIMIENTO CON GLOVE (PROMEDIADO) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_glove):.4f}")

## 2. Embeddings Contextuales: El fenómeno BERT
Los modelos estáticos tienen un fallo: la **polisemia**. La palabra "banco" recibe el mismo vector si hablas de dinero o de un parque.

**BERT** (Bidirectional Encoder Representations from Transformers) soluciona esto generando vectores **dinámicos**: el vector de una palabra cambia según las palabras que tiene alrededor.



### Conceptos Clave de BERT:
1. **WordPiece Tokenization:** Descompone palabras raras en sub-unidades (ej: "playing" -> "play", "##ing") para evitar palabras desconocidas.
2. **Token [CLS]:** BERT añade este token al inicio. Su vector de salida representa el "resumen" de toda la oración, ideal para clasificación.

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

# Configuración de hardware (Uso de GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cargamos el tokenizador y el modelo base de BERT
tok = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased").to(device)
bert.eval() # Modo evaluación (no entrenamiento)

def bert_cls_embedding(texts, batch_size=32, max_length=128):
    """
    Extrae el embedding del token [CLS] para cada texto usando BERT.
    """
    embs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        # Tokenización y conversión a tensores de PyTorch
        enc = tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        
        with torch.no_grad():
            enc = {k: v.to(device) for k, v in enc.items()}
            out = bert(**enc).last_hidden_state  # Forma: [Batch, Longitud, Hidden_Size]
            # Extraemos solo el primer token [CLS] (índice 0)
            cls_embeddings = out[:, 0, :].cpu().numpy()
            embs.append(cls_embeddings)
            
    return np.vstack(embs)

# Nota: Este proceso es costoso computacionalmente
X_bert = bert_cls_embedding(df["text"].tolist())

# División de datos específica para BERT
Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(X_bert, df['label_num'], test_size=0.2, random_state=123)

# Clasificación final
clf_bert = LogisticRegression(max_iter=2000)
clf_bert.fit(Xtr_b, ytr_b)
y_pred_bert = clf_bert.predict(Xte_b)

print("\n--- RENDIMIENTO CON BERT ([CLS] EMBEDDINGS) ---")
print(f"Accuracy: {accuracy_score(yte_b, y_pred_bert):.4f}")
print(classification_report(yte_b, y_pred_bert))

# Ejercicio


**Reto: Clasificación Híbrida de Mensajes**

Has aprendido que las **características manuales** son excelentes para capturar la "forma" del mensaje, mientras que **TF-IDF** captura la "esencia" de las palabras. 

**Tu objetivo:** Implementar un modelo que combine ambas técnicas y comparar si esta "unión de fuerzas" supera a los modelos por separado.

---

### Instrucciones del Ejercicio:
1. **Característica Manual Nueva:** Crea una función que cuente cuántos signos de exclamación (`!`) tiene cada mensaje. (Hipótesis: El spam abusa de la exclamación para crear urgencia).
2. **Representación Vectorial:** Genera una matriz TF-IDF de los mensajes.
3. **Modelo Híbrido:** Concatena (une) el conteo de exclamaciones con la matriz TF-IDF.
4. **Evaluación:** Compara el Accuracy del modelo híbrido contra el modelo que solo usa TF-IDF.

### 1. Preparación del "Laboratorio"
Usaremos el dataset ya cargado en las partes anteriores.

In [ ]:
from scipy.sparse import hstack # Para unir matrices dispersas con vectores

### 2. Implementación del Ejercicio

In [ ]:
# TAREA 1: Crear la característica de exclamaciones
# dataframe['exclamation_count'] = ...

# TAREA 2: Crear matriz TF-IDF (puedes usar el vectorizador ya entrenado)
# X_tfidf = ...

# TAREA 3: Unir (hstack) la característica manual con la matriz TF-IDF
# 
# X_combined = hstack([X_tfidf, dataframe[['exclamation_count']].values])